# CV Masterclass 18: Unsupervised Anomaly Detection

Welcome to Notebook 18. In previous modules, we mastered supervised tasks: classifying cats, detecting pedestrians, and segmenting tumors. However, in industrial AI, medicine, and security, we often face a paradox: **we know what "normal" looks like, but we can never predict every way "abnormal" might manifest.**

Today, we move beyond labeling and into **Unsupervised Representation Learning**. We will build systems that learn the manifold of "Normal" and flag anything that deviates from it without ever having seen a single defect during training.

---

## 🎓 Socratic Deep Check
Ponder these questions before we begin:

> *"If you train a classifier to distinguish between 'A' and 'B', but then show it a 'C', why is it more likely to confidently misclassify it as 'A' or 'B' rather than saying 'I don't know'?"*

> *"Why are coarse, mid-level features (like textures and edges) often more useful for defect detection than the high-level semantic features (like 'wheel' or 'face') found in the final layers of a CNN?"*

---

## Table of Contents
1. **The Flaw of Supervised Defect Detection:** Why Class Imbalance Kills Standard Models.
2. **PatchCore: The SOTA Paradigm:** Feature Extraction, Memory Banks, and Coresets.
3. **Mathematics of Anomaly Scoring:** Nearest Neighbor Distance and Heatmap Generation.
4. **Student-Teacher Networks:** Detecting Anomalies via Mimicry Error.
5. **Implementation:** Building a PatchCore Feature Extractor in PyTorch.
6. **Common Pitfalls:** Overfitting to Noise and Calibration Challenges.

## 1. The Flaw of Supervised Defect Detection

In a typical manufacturing pipeline (e.g., inspecting phone screens), you might have 1,000,000 "Perfect" samples and only 5 "Scratched" samples. This presents two critical failures for supervised learning:

### A. The Extreme Imbalance Problem
Standard Cross-Entropy Loss assumes a relatively balanced class distribution. When $P(y=normal) = 0.999995$, the gradient from the defect class is negligible. Even with oversampling or Focal Loss, you are training the model to recognize *specific* defects it has seen, not the concept of "defectiveness."

### B. The "Unknown Unknowns"
Supervised models are closed-world. If you train a model on 'Scratches' and 'Dents', and tomorrow a 'Discoloration' defect appears, the model will try to map it to 'Scratch' or 'Dent' because it has no mathematical concept of **OOD (Out-of-Distribution)**.

**The Unsupervised Solution:** We define normality. Anything that does not fit the statistical distribution of normality is an anomaly.
$$ \text{Anomaly Score}(x) = -\log P(x | \theta_{\text{normal}}) $$

## 2. PatchCore: The State of the Art

PatchCore (Roth et al., 2021) is the current gold standard for industrial anomaly detection. It works by capturing a "Memory Bank" of locally-aware patch features from a frozen, pre-trained backbone.

### Why Frozen Backbones?
Pre-trained models (like WideResNet-50 on ImageNet) have already learned a rich hierarchy of visual features. We don't need to retrain them; we just need to learn what *portions* of their feature space represent our specific "Normal" data.

### The Pipeline:
1.  **Extract:** Pass normal images through a ResNet. Save feature maps from middle layers (e.g., Layer 2 and Layer 3).
2.  **Pool:** Use Adaptive Average Pooling to create "Locally Aware" patches. This ensures that a single feature vector captures context from its neighbors.
3.  **Aggregate:** Store all these patch vectors in a **Memory Bank** $\mathcal{M}$.
4.  **Coreset Subsampling:** Since storing every patch from every image is memory-intensive, we use a greedy $k$-center algorithm to select a representative subset that covers the same feature space.

## 3. Mathematics of Anomaly Scoring

Given a test image, we extract its patch features $\mathcal{F}_{test}$. For each patch $f \in \mathcal{F}_{test}$, we calculate its distance to the nearest neighbor in our Memory Bank $\mathcal{M}$:

$$ s(f) = \min_{m \in \mathcal{M}} \| f - m \|_2 $$

### Image-Level Score
The total anomaly score for the image is the maximum distance found across all patches, often scaled to be more robust:
$$ S_{image} = \max_{f \in \mathcal{F}_{test}} s(f) $$

### Anomaly Heatmap
By plotting the distance $s(f)$ for every spatial location $(i, j)$ and upsampling back to the original image size, we get a precise localization of the defect.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

class PatchCoreExtractor(nn.Module):
    """
    Extracts locally-aware patch features from mid-level ResNet layers.
    """
    def __init__(self):
        super().__init__()
        self.backbone = models.wideresnet50_2(pretrained=True)
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.backbone.eval()

        # We use Layer 2 and Layer 3 features
        self.features = []
        self.backbone.layer2[-1].register_forward_hook(self.hook)
        self.backbone.layer3[-1].register_forward_hook(self.hook)

    def hook(self, module, input, output):
        self.features.append(output)

    def forward(self, x):
        self.features = []
        _ = self.backbone(x)
        
        # Globally average pool or locally pool features
        # Here we demonstrate spatial alignment
        pooled_features = []
        for f in self.features:
            # Locally aware pooling (e.g., 3x3 avg pool)
            p = F.avg_pool2d(f, kernel_size=3, stride=1, padding=1)
            pooled_features.append(p)
        
        # Resize to match Layer 2 dimensions and concatenate
        target_size = pooled_features[0].shape[-2:]
        resized = [F.interpolate(f, size=target_size, mode='bilinear', align_corners=False) for f in pooled_features]
        
        # Result: [Batch, Channels, H, W]
        return torch.cat(resized, dim=1)

def calculate_anomaly_score(test_features, memory_bank):
    """
    Simplified scoring logic: Euclidean distance to nearest neighbor.
    """
    # Flatten features to [N_patches, D]
    B, C, H, W = test_features.shape
    test_patches = test_features.permute(0, 2, 3, 1).reshape(-1, C)
    
    # Calculate pair-wise distances (Using torch.cdist for efficiency)
    # In practice, Memory Bank M is very large, so we'd use FAISS or Coreset sampling.
    distances = torch.cdist(test_patches, memory_bank) # [N_test, N_memory]
    
    # Distance to nearest neighbor for each patch
    min_distances, _ = torch.min(distances, dim=1)
    
    # Reshape back to image grid
    anomaly_map = min_distances.reshape(B, H, W)
    image_score = torch.max(anomaly_map)
    
    return anomaly_map, image_score

# TEST IT
extractor = PatchCoreExtractor()
dummy_input = torch.randn(1, 3, 224, 224)
features = extractor(dummy_input)
print(f"Feature Shape: {features.shape}") # Expected (1, 1536, 28, 28)

# Simulate a tiny Memory Bank of 'Normal' features
memory_bank = torch.randn(100, features.shape[1])
anomaly_map, score = calculate_anomaly_score(features, memory_bank)

assert anomaly_map.shape == (1, 28, 28)
assert score > 0
print("PatchCore Logic Verified!")

## 4. Student-Teacher Networks

A different but equally powerful approach is **Knowledge Distillation** for anomaly detection.

### The Setup
1.  **Teacher ($T$):** A frozen pre-trained model (ResNet).
2.  **Student ($S$):** A model with identical architecture, trained to mimic the Teacher's output on **Normal samples only**.

### The Mimicry Gap
Since the Student only sees "Normal" data, it learns the mapping from pixel space to the Teacher's embedding space for normal patterns. When an anomaly (e.g., a crack) appears:
- The **Teacher** extracts features normally (representing the crack).
- The **Student** fails to reconstruct those features because it has never generalized to that specific out-of-distribution manifold.

The **Residual** (Difference) between $T(x)$ and $S(x)$ provides the anomaly score.
$$ \text{Error}(x) = \| T(x) - S(x) \|^2 $$

### COMMON PITFALLS
*   **Layer Selection:** Using the very last layer of a CNN often fails. Why? Because the final layer is highly semantic (it sees "Table" or "Dog"). If the 'Table' has a tiny scratch, the final layer might ignore it. Mid-level layers are sensitive to local structural changes.
*   **Feature Drifting:** If your camera shifts slightly or the lighting changes, the features will change. Your memory bank must be robust to these variations, often requiring "Multimodal Normality" (multiple lighting conditions).
*   **Curse of Dimensionality:** Patch vectors can be huge (e.g., 1536-D). Calculating distances in high dimensions can be slow and noisy. Dimensionality reduction (PCA or Gaussian Random Projection) is often applied to the memory bank.

### 🎓 DEEP QUESTION ANSWERED

**Q:** *If you train a classifier to distinguish between 'A' and 'B', why is it more likely to confidently misclassify it as 'A' or 'B' rather than saying 'I don't know'?*

**A:** This is the **Softmax Calibration Problem**. Softmax only cares about the relative differences between logit values. If an image is totally foreign, it might still produce a slightly higher score for 'A' than 'B', which after an exponential and normalization becomes high confidence. Unsupervised anomaly detection solves this by measuring the **Absolute Distance** to known data, rather than relative probability.

**Q:** *Why are coarse, mid-level features better for defect detection?*

**A:** Because defects are usually local structural violations (texture breaks, color shifts, geometric anomalies). High-level features are designed to be **invariant** to these details (e.g., a cat is still a cat even if it has a small scar). In anomaly detection, we don't want invariance; we want **sensitivity** to local deviations.